In [1]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
# Step 1: Set coordinates for the Al-Hasakah agricultural sector in Syria
LATITUDE = 36.5024
LONGITUDE = 40.7476

def fetch_syria_climate_data():
    # Use your exact verified manual URL link
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={LATITUDE}&longitude={LONGITUDE}&start_date=2024-01-01&end_date=2025-12-31&daily=precipitation_sum,temperature_2m_max,soil_moisture_7_to_28cm_mean,soil_moisture_28_to_100cm_mean&timezone=Europe%2FMoscow"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        daily_data = data['daily']
        
        # Read the exact JSON dictionary keys output by your custom manual URL parameter string
        df = pd.DataFrame({
            'date': pd.to_datetime(daily_data['time']),
            'max_temp': daily_data['temperature_2m_max'],
            'precipitation': daily_data['precipitation_sum'],
            'root_zone_moisture': daily_data['soil_moisture_7_to_28cm_mean'],
            'deep_soil_moisture': daily_data['soil_moisture_28_to_100cm_mean']
        })
        return df
    else:
        raise Exception(f"API Data Stream Connection Failure: Status Code {response.status_code}")

def process_data(df):
    # Forward-fill any missing records to guarantee data pipeline reliability
    df = df.ffill()
    
    # Calculate a 14-day rolling precipitation deficit
    df['rolling_rain_14d'] = df['precipitation'].rolling(window=14, min_periods=1).sum()
    
    # Target Label: Flag Severe Drought Risk (1) based on compound environmental indicators
    df['drought_risk_label'] = np.where(
        (df['root_zone_moisture'] < 0.22) & (df['deep_soil_moisture'] < 0.25) & (df['max_temp'] > 34), 1, 0
    )
    return df

if __name__ == "__main__":
    print("🚀 Ingesting raw climate telemetry data for Syria...")
    raw_df = fetch_syria_climate_data()
    
    print("🛠️ Running feature engineering pipelines...")
    clean_df = process_data(raw_df)
    
    clean_df.to_csv("processed_climate_data.csv", index=False)
    print("✅ Pipeline executed successfully! Output saved to 'processed_climate_data.csv'")


🚀 Ingesting raw climate telemetry data for Syria...
🛠️ Running feature engineering pipelines...
✅ Pipeline executed successfully! Output saved to 'processed_climate_data.csv'
